In [ ]:
import brightway2 as bw
from premise import *
import os
import wurst
import time
import copy

In [ ]:
startTime = time.time() # just to see how much time the code takes to run (this is the start)

In [ ]:
bw.projects.set_current('Prospective LCA v1') # set current project

In [ ]:
list(bw.databases)

In [ ]:
bioSphereDBName = 'biosphere3'
bioSphereDB = bw.Database(bioSphereDBName) # import the biosphere database
ecoInventBase2020DBName = 'ecoinvent 3.8 cutoff image SSP2-Base 2020'
ecoInventBase2020DB = bw.Database(ecoInventBase2020DBName) # import the ecoinvent database (image base 2020)

In [ ]:
# import base inventories from excel
excelImport = bw.ExcelImporter(os.path.join('..', 'Raw data', 'lci-Abhi image SSP2-Base 2020.xlsx'))
excelImport.apply_strategies()
excelImport.match_database(ecoInventBase2020DBName, fields = ('name', 'reference product', 'unit', 'location')) # match flows from ecoinvent database
excelImport.match_database(bioSphereDBName, fields = ('name', 'unit', 'categories')) # match flows from biosphere database
excelImport.match_database(fields = ('name', 'reference product', 'unit', 'location')) # match flows from my database
excelImport.statistics()

In [ ]:
# modify inventories to substitute 'reference product' with 'product' in all the exchanges
for ds in excelImport:
    for exchange in ds['exchanges']:
        if 'reference product' in exchange:
            exchange['product'] = exchange.pop('reference product')

In [ ]:
renewableElectricityInventories = [activity for activity in excelImport if 'renewable electricity' in activity['reference product']
                                     and 'Abhi' in activity['reference product']] # all renewable electricity inventories
carbonDioxideInventories = [activity for activity in excelImport if 'carbon dioxide' in activity['reference product']
                                                                and 'Abhi' in activity['reference product']] # all carbon dioxide inventories
hydrogenInventories = [activity for activity in excelImport if 'hydrogen' in activity['reference product']
                                                                and 'Abhi' in activity['reference product']] # all hydrogen inventories
baseGAmmoniaInventory = [activity for activity in excelImport if 'ammonia' in activity['reference product']
                                    and 'Abhi' in activity['reference product']
                                    and 'BAU' not in activity['name']
                                    and 'biomethane' not in activity['name']
                                    and 'hydrogen' in activity['name']][0] # base green ammonia inventory
baseGMethanolInventory = [activity for activity in excelImport if 'methanol' in activity['reference product']
                                    and 'Abhi' in activity['reference product']
                                    and 'BAU' not in activity['name']
                                    and 'biomethane' not in activity['name']][0] # base green methanol inventory
baseMTOEthyleneInventory = [activity for activity in excelImport if 'ethylene' in activity['reference product']
                                    and 'Abhi' in activity['reference product']
                                    and 'methanol' in activity['name']
                                    and 'BAU' in activity['name']
                                    and 'biogas' not in activity['name'] 
                                    and 'electroreduction' not in activity['name']][0] # base ethylene inventory via MTO
baseElectroEthyleneInventory = [activity for activity in excelImport if 'ethylene' in activity['reference product']
                                    and 'Abhi' in activity['reference product']
                                    and 'methanol' not in activity['name']
                                    and 'electroreduction' in activity['name']][0] # base ethylene inventory via direct electroreduction of carbon dioxide

In [ ]:
lciBase2020DBName = 'lci-Abhi image SSP2-Base 2020'

In [ ]:
gAmmoniaInventories = []

# creating new green ammonia inventories from different hydrogen inventories
for hydrogenInventory in hydrogenInventories:
    newInventory = copy.deepcopy(baseGAmmoniaInventory)
    newInventory.update({'name' : 'ammonia; ' + hydrogenInventory['name']})
    newInventory.update({'code' : wurst.filesystem.get_uuid()})
    for exchange in newInventory['exchanges']:
        if exchange['database'] == lciBase2020DBName and exchange['type'] == 'technosphere':
            if exchange['product'] == 'hydrogen, Abhi':
                exchange.update({'name' : hydrogenInventory['name']})
        elif exchange['database'] == lciBase2020DBName and exchange['type'] == 'production':
                exchange.update({'name' : newInventory['name']})
    gAmmoniaInventories.append(newInventory)

# deleteing the base greem ammonia inventory to avoid duplicates
for i, d in enumerate(excelImport.data):
    if d == baseGAmmoniaInventory:
        excelImport.data.pop(i)

In [ ]:
gMethanolInventories = []

# creating new green methanol inventories from different carbon dioxide and hydrogen inventories
for carbonDioxideInventory in carbonDioxideInventories:
    for hydrogenInventory in hydrogenInventories:
        newInventory = copy.deepcopy(baseGMethanolInventory)
        newInventory.update({'name' : 'methanol; ' + carbonDioxideInventory['name'] + '; ' + hydrogenInventory['name']})
        newInventory.update({'code' : wurst.filesystem.get_uuid()})
        for exchange in newInventory['exchanges']:
            if exchange['database'] == lciBase2020DBName and exchange['type'] == 'technosphere':
                if exchange['product'] == 'carbon dioxide, Abhi':
                    exchange.update({'name' : carbonDioxideInventory['name']})
                elif exchange['product'] == 'hydrogen, Abhi':
                    exchange.update({'name' : hydrogenInventory['name']})
            elif exchange['database'] == lciBase2020DBName and exchange['type'] == 'production':
                exchange.update({'name' : newInventory['name']})
        gMethanolInventories.append(newInventory)

# deleteing the base greem methanol inventory to avoid duplicates
for i, d in enumerate(excelImport.data):
    if d == baseGMethanolInventory:
        excelImport.data.pop(i)

In [ ]:
# make more efficient
allMethanolInventories = [activity for activity in (excelImport.data + gMethanolInventories) if 'methanol' in activity['reference product']
                                     and 'Abhi' in activity['reference product']] # all methanol inventories (both created and imported)

In [ ]:
MTOEthyleneInventories = []

# creating new ethylene inventories via MTO from different methanol inventories
for methanolInventory in allMethanolInventories:
    newInventory = copy.deepcopy(baseMTOEthyleneInventory)
    if 'BAU' in methanolInventory['name']:
        newInventory.update({'name' : 'ethylene, fMTO; ' + methanolInventory['name']})
    elif 'biomethane' in methanolInventory['name'] and 'hydrogen' not in methanolInventory['name']:
        newInventory.update({'name' : 'ethylene, bmMTO; ' + methanolInventory['name']})
    else:
        newInventory.update({'name' : 'ethylene, gMTO; ' + methanolInventory['name']})
    newInventory.update({'code' : wurst.filesystem.get_uuid()})
    for exchange in newInventory['exchanges']:
        if exchange['database'] == lciBase2020DBName and exchange['type'] == 'technosphere':
            if exchange['product'] == 'methanol, Abhi':
                exchange.update({'name' : methanolInventory['name']})
        elif exchange['database'] == lciBase2020DBName and exchange['type'] == 'production':
                exchange.update({'name' : newInventory['name']})
    MTOEthyleneInventories.append(newInventory)

# deleteing the base ethylene inventory via MTO to avoid duplicates
for i, d in enumerate(excelImport.data):
    if d == baseMTOEthyleneInventory:
        excelImport.data.pop(i)

In [ ]:
electroEthyleneInventories = []

# creating new ethylene inventories via direct electroreduction of carbon dioxide from different carbon dioxide and electricity inventories
for carbonDioxideInventory in carbonDioxideInventories:
    for renewableElectricityInventory in renewableElectricityInventories:
        newInventory = copy.deepcopy(baseElectroEthyleneInventory)
        newInventory.update({'name' : 'ethylene, electroreduction; ' + carbonDioxideInventory['name'] + '; ' + renewableElectricityInventory['name']})
        newInventory.update({'code' : wurst.filesystem.get_uuid()})
        for exchange in newInventory['exchanges']:
            if exchange['database'] == lciBase2020DBName and exchange['type'] == 'technosphere':
                if exchange['product'] == 'carbon dioxide, Abhi':
                    exchange.update({'name' : carbonDioxideInventory['name']})
                elif exchange['product'] == 'renewable electricity, Abhi':
                    exchange.update({'name' : renewableElectricityInventory['name']})
            elif exchange['database'] == lciBase2020DBName and exchange['type'] == 'production':
                exchange.update({'name' : newInventory['name']})
        electroEthyleneInventories.append(newInventory)

# deleteing the base ethylene inventory via direct electroreduction of carbon dioxide to avoid duplicates
for i, d in enumerate(excelImport.data):
    if d == baseElectroEthyleneInventory:
        excelImport.data.pop(i)

In [ ]:
# creating new ethylene inventories via direct electroreduction of carbon dioxide including 25 percent replacement of electrolytes
"""ALLOC_FACTOR = 0.96 
KOH = 0.0382 
WASTE_WATER = -0.0556"""

for electroEthyleneInventory in electroEthyleneInventories:
    newInventory = copy.deepcopy(electroEthyleneInventory)
    newInventory.update({'name' : electroEthyleneInventory['name'] + '; 25 percent replacement'})
    newInventory.update({'code' : wurst.filesystem.get_uuid()})
    for exchange in newInventory['exchanges']:
        if exchange['database'] == ecoInventBase2020DBName and exchange['type'] == 'technosphere':
            if 'potassium hydroxide' in exchange['product']:
                exchange.update({'amount' : 0.0382 * 0.96}) # from Iasonas et al. (ethylene) # define parameters before
            elif 'wastewater, average' in exchange['product']:
                exchange.update({'amount' : -0.0556 * 0.96}) # from Iasonas et al. (ethylene) # define parameters before
        elif exchange['database'] == lciBase2020DBName and exchange['type'] == 'production':
                exchange.update({'name' : newInventory['name']})
    electroEthyleneInventories = electroEthyleneInventories + [newInventory]

In [ ]:
globalDB = excelImport.data + gAmmoniaInventories + gMethanolInventories + MTOEthyleneInventories + electroEthyleneInventories # add all inventories
ecoInventBase2020DBDict = [ds.as_dict() for ds in ecoInventBase2020DB] # convert ecoinvent database to dictionary
bioSphereDBDict = [ds.as_dict() for ds in bioSphereDB] # convert biosphere database to dictionary # maybe not needed?

newLocations = {'CN' : 'China',
                'IN' : 'India',
                'RER' : 'Europe',
                'US' : 'United States',
                'CH' : 'Switzerland',
                'DE' : 'Germany',
                'GB' : 'United Kingdom',
                'ZA' : 'South Africa',
                'AU' : 'Australia',
                'RU' : 'Russia'}

DSToRegionalize = globalDB

regionalizedDS = []
for ds in DSToRegionalize:
    for loc in newLocations:
        dsCopy = wurst.transformations.geo.copy_to_new_location(ds, loc)
        regionalizedDS.append(dsCopy)


# Relink technosphere exchanges to the new locations
for ds in regionalizedDS:
    for exchange in ds['exchanges']:
        if exchange['type'] == 'technosphere':
            if exchange['database'] == ecoInventBase2020DBName:
                dsOutput = [dsInstance for dsInstance in ecoInventBase2020DBDict if exchange['name'] == dsInstance['name'] 
                                and exchange['product'] == dsInstance['reference product'] 
                                and ds['location'] == dsInstance['location']]
                if not dsOutput and 'market group for electricity' in exchange['name']:
                    exchange['name'] = exchange['name'].replace('group ', '')
                    dsOutput = [dsInstance for dsInstance in ecoInventBase2020DBDict if exchange['name'] == dsInstance['name'] 
                                and exchange['product'] == dsInstance['reference product'] 
                                and ds['location'] == dsInstance['location']]
            elif exchange['database'] == lciBase2020DBName:
                dsOutput = [dsInstance for dsInstance in regionalizedDS if exchange['name'] == dsInstance['name']
                                and ds['location'] == dsInstance['location']]
            if dsOutput:
                    exchange.update({'location' : dsOutput[0]['location']})

In [ ]:
db = globalDB + regionalizedDS
DBLinked = copy.deepcopy(db)

production = lambda x : x['type'] == 'production'
technosphere = lambda x : x['type'] == 'technosphere'
biosphere = lambda x : x['type'] == 'biosphere'

# linking exchanges to database inventories
for ds in DBLinked:
    for exchange in filter(production, ds['exchanges']):
        exchange.update({'input' : (ds['database'], ds['code'])})
    for exchange in filter(technosphere, ds['exchanges']):
        try:
            exchangeLink = wurst.get_one(db + ecoInventBase2020DBDict,
                                         wurst.equals('name', exchange['name']),
                                         wurst.equals('reference product', exchange['product']),
                                         wurst.equals('location', exchange['location']))
            exchange.update({'input' : (exchangeLink['database'], exchangeLink['code'])})
        except Exception:
            print(exchange['name'], exchange['product'], exchange['location'])
            raise
    # biosphere links maybe not needed
    for exchange in filter(biosphere, ds['exchanges']):
        if 'input' not in exchange:
            try:
                exchangeLink = [ds['code'] for ds in bioSphereDBDict if ds['name'] == exchange['name'] and
                                                                        ds['unit'] == exchange['unit'] and
                                                                        ds['categories'] == exchange['categories']][0]
                exchange.update({'input' : (bioSphereDBName, exchangeLink)})
            except Exception:
                print(exchange['name'], exchange['unit'], exchange['categories'])
                raise

In [ ]:
if lciBase2020DBName in bw.databases:
    del bw.databases[lciBase2020DBName]  
wurst.write_brightway2_database(DBLinked, lciBase2020DBName) # write database

In [ ]:
lciBase2020DB = wurst.extract_brightway2_databases(lciBase2020DBName)

In [ ]:
premiseDBNames = ['image SSP2-Base 2030',
                  'image SSP2-Base 2040',
                  'image SSP2-Base 2050',
                  'image SSP2-RCP19 2030',
                  'image SSP2-RCP19 2040',
                  'image SSP2-RCP19 2050',
                  'image SSP2-RCP26 2030',
                  'image SSP2-RCP26 2040',
                  'image SSP2-RCP26 2050',
                  'remind SSP1-PkBudg500 2030',
                  'remind SSP1-PkBudg500 2040',
                  'remind SSP1-PkBudg500 2050',
                  'remind SSP1-PkBudg1150 2030',
                  'remind SSP1-PkBudg1150 2040',
                  'remind SSP1-PkBudg1150 2050',
                  'remind SSP2-Base 2030',
                  'remind SSP2-Base 2040',
                  'remind SSP2-Base 2050',
                  'remind SSP2-PkBudg500 2030',
                  'remind SSP2-PkBudg500 2040',
                  'remind SSP2-PkBudg500 2050',
                  'remind SSP2-PkBudg1150 2030',
                  'remind SSP2-PkBudg1150 2040',
                  'remind SSP2-PkBudg1150 2050',
                  'remind SSP5-PkBudg500 2030',
                  'remind SSP5-PkBudg500 2040',
                  'remind SSP5-PkBudg500 2050',
                  'remind SSP5-PkBudg1150 2030',
                  'remind SSP5-PkBudg1150 2040',
                  'remind SSP5-PkBudg1150 2050']

In [ ]:
for premiseDBName in premiseDBNames:
    print('linking database ' + premiseDBName + '...')
    premiseDBDict = [ds.as_dict() for ds in bw.Database('ecoinvent 3.8 cutoff ' + premiseDBName)]

    DBLinked = copy.deepcopy(lciBase2020DB)

    production = lambda x : x['type'] == 'production'  
    technosphere = lambda x : x['type'] == 'technosphere'
    biosphere = lambda x : x['type'] == 'biosphere'

    for ds in DBLinked:
        for exchange in filter(technosphere, ds['exchanges']):
            try:
                exchangeLink = wurst.get_one(lciBase2020DB + premiseDBDict,
                                            wurst.equals('name', exchange['name']),
                                            wurst.equals('reference product', exchange['product']),
                                            wurst.equals('location', exchange['location'])
                                            )
                exchange.update({'input' : (exchangeLink['database'], exchangeLink['code'])})
                exchange.update({'database' : exchangeLink['database']})
            except Exception:
                print(exchange['name'], exchange['reference product'], exchange['location'])
                raise
        for exchange in filter(biosphere, ds['exchanges']):
            if 'input' not in exchange:
                try:
                    exchangeLink = [ds['code'] for ef in bioSphereDBDict if ds['name'] == exchange['name'] and 
                                                                            ds['unit'] == exchange['unit'] and 
                                                                            ds['categories'] == exchange['categories']][0]
                    exchange.update({'input': ('biosphere3', exchangeLink)})   
                except Exception:
                    print(exchange['name'], exchange['unit'], exchange['categories'])
                    raise
    dbName = 'lci-Abhi ' + premiseDBName
    if dbName in bw.databases:
        del bw.databases[dbName]
    wurst.write_brightway2_database(DBLinked, dbName)
    print(premiseDBName + ' linking and writing successful!')


In [ ]:
endTime = time.time() # end time
elapsedTime = endTime - startTime # calculate elapsed time
print(f'Elapsed time: {elapsedTime/3600:.2f} hours') 